# CE 497 — Sea-Ice Segmentation with Segment Anything Model (SAM)

In this notebook, we use Meta's **Segment Anything Model (SAM)** to automatically segment sea-ice imagery.
For course reproducibility, the SAM source code and the ViT-B pretrained checkpoint are archived under the **CE497_AI-Civil** GitHub repository, so the notebook does not need to download SAM from Meta/Facebook at runtime.

We compare two approaches:

### Trial 1 — Segment the entire image at once
SAM sees the complete image and automatically generates masks.

### Trial 2 — Divide the image into an \(N \times N\) grid
We divide each image dimension into **N equal parts**, run SAM independently on every tile, and then stitch all tile masks back into the original full-image coordinates.

The motivation is that small sea-ice floes may be difficult for SAM to resolve when the whole image is resized for the model. Tiling makes small floes occupy a larger fraction of SAM's input.

> **No manual point prompt, bounding-box prompt, or text prompt is used.**  
> Both trials use `SamAutomaticMaskGenerator`.

## Before you begin

In Google Colab choose:

**Runtime → Change runtime type → T4 GPU**

Then:

**Runtime → Run all**


## 1. Install the CE497 archived copy of Segment Anything

The SAM source code is stored in `CE497_AI-Civil/third_party/segment-anything`.
We install the frozen `sam-v1` tag rather than downloading the package from Meta's GitHub repository.


In [ ]:
# Install the archived SAM source from the CE497 repository.
# The @sam-v1 tag freezes the exact source version used by this notebook.
!pip -q install "git+https://github.com/olivmeng/CE497_AI-Civil.git@sam-v1#subdirectory=third_party/segment-anything"
!pip -q install opencv-python matplotlib


## 2. Import packages and check the GPU

In [ ]:
import os
import urllib.request
import time

import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch

from segment_anything import sam_model_registry, SamAutomaticMaskGenerator

device = "cuda" if torch.cuda.is_available() else "cpu"

print("PyTorch version:", torch.__version__)
print("Device:", device)

if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: SAM will be much slower on CPU.")
    print("In Colab, select Runtime → Change runtime type → T4 GPU.")

## 3. Download the sea-ice image from the CE 497 GitHub repository

The image is stored at:

`CE497_AI-Civil/Figures/FrontierPaper_sea ice.png`

The notebook downloads it automatically.


In [ ]:
IMAGE_URL = (
    "https://raw.githubusercontent.com/"
    "olivmeng/CE497_AI-Civil/main/"
    "Figures/FrontierPaper_sea%20ice.png"
)

IMAGE_PATH = "/content/sea_ice.png"

urllib.request.urlretrieve(IMAGE_URL, IMAGE_PATH)

print("Downloaded image to:", IMAGE_PATH)

## 4. Load and display the original image

In [ ]:
image_bgr = cv2.imread(IMAGE_PATH)

if image_bgr is None:
    raise RuntimeError("The image could not be loaded.")

# OpenCV reads BGR; SAM expects RGB.
image = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

H, W = image.shape[:2]

# ---------------------------------------------------------
# Physical scale of the satellite image
# ---------------------------------------------------------
# The full image width represents 1500 m in the real world.
# If you use a different image later, change only this value.
IMAGE_WIDTH_M = 1500.0

# Assume square image pixels.
meters_per_pixel = IMAGE_WIDTH_M / W
pixel_area_m2 = meters_per_pixel**2

# Physical image height follows from the image aspect ratio.
IMAGE_HEIGHT_M = H * meters_per_pixel

print(f"Image size:       width={W}, height={H} pixels")
print(f"Physical width:   {IMAGE_WIDTH_M:.2f} m")
print(f"Physical height:  {IMAGE_HEIGHT_M:.2f} m")
print(f"Image resolution: {meters_per_pixel:.6f} m/pixel")
print(f"Pixel area:       {pixel_area_m2:.6f} m^2/pixel")

plt.figure(figsize=(10, 10))
plt.imshow(image)
plt.title(f"Original Sea-Ice Image — {IMAGE_WIDTH_M:.0f} m wide")
plt.axis("off")
plt.show()


## 5. Download the archived pretrained SAM checkpoint

We use the original SAM **ViT-B** checkpoint (`sam_vit_b_01ec64.pth`) because it is lighter than ViT-L and ViT-H and is convenient for a classroom Colab exercise.

For course reproducibility, the checkpoint is downloaded from the `sam-v1` GitHub Release of **CE497_AI-Civil**, rather than from Meta's download server. The SHA-256 hash is checked after download to make sure the file has not been corrupted or replaced.


In [ ]:
import hashlib

SAM_CHECKPOINT_URL = (
    "https://github.com/olivmeng/CE497_AI-Civil/"
    "releases/download/sam-v1/sam_vit_b_01ec64.pth"
)

SAM_CHECKPOINT = "/content/sam_vit_b_01ec64.pth"
EXPECTED_SHA256 = "ec2df62732614e57411cdcf32a23ffdf28910380d03139ee0f4fcbe91eb8c912"

if not os.path.exists(SAM_CHECKPOINT):
    print("Downloading archived SAM ViT-B checkpoint from CE497_AI-Civil...")
    urllib.request.urlretrieve(SAM_CHECKPOINT_URL, SAM_CHECKPOINT)

sha256 = hashlib.sha256()
with open(SAM_CHECKPOINT, "rb") as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b""):
        sha256.update(chunk)

actual_sha256 = sha256.hexdigest()

if actual_sha256 != EXPECTED_SHA256:
    raise RuntimeError(
        "SAM checkpoint SHA-256 mismatch. "
        f"Expected {EXPECTED_SHA256}, got {actual_sha256}."
    )

print("SAM checkpoint:", SAM_CHECKPOINT)
print("SHA-256 verified:", actual_sha256)


## 6. Load SAM

In [ ]:
MODEL_TYPE = "vit_b"

sam = sam_model_registry[MODEL_TYPE](checkpoint=SAM_CHECKPOINT)
sam.to(device=device)
sam.eval()

print("SAM loaded successfully.")

## 7. Helper functions

In [ ]:
def make_mask_generator(model):
    """
    Create SAM's automatic mask generator.
    """
    return SamAutomaticMaskGenerator(
        model=model,
        points_per_side=48,
        pred_iou_thresh=0.85,
        stability_score_thresh=0.90,
        crop_n_layers=1,
        crop_n_points_downscale_factor=2,
        min_mask_region_area=30,
    )


def create_mask_overlay(image_rgb, masks, alpha=0.62, random_seed=42):
    """
    Overlay automatically generated masks on an RGB image.
    """
    rng = np.random.default_rng(random_seed)
    result = image_rgb.astype(np.float32).copy()

    # Draw large masks first so small masks remain visible on top.
    masks_sorted = sorted(masks, key=lambda x: x["area"], reverse=True)

    for mask_info in masks_sorted:
        mask = mask_info["segmentation"]
        color = rng.integers(0, 256, size=3)

        result[mask] = (
            (1.0 - alpha) * result[mask]
            + alpha * color
        )

    return np.clip(result, 0, 255).astype(np.uint8)


def masks_to_label_image(masks, image_shape):
    """
    Convert a list of SAM masks into one integer label image.

    0 = unlabeled/background
    1, 2, 3, ... = detected masks
    """
    height, width = image_shape[:2]
    label_image = np.zeros((height, width), dtype=np.uint16)

    masks_sorted = sorted(masks, key=lambda x: x["area"], reverse=True)

    for object_id, mask_info in enumerate(masks_sorted, start=1):
        label_image[mask_info["segmentation"]] = object_id

    return label_image


def show_three_panel(original, overlay, labels, title):
    """
    Show the original image, colored SAM masks, and integer label mask.
    """
    plt.figure(figsize=(20, 7))

    plt.subplot(1, 3, 1)
    plt.imshow(original)
    plt.title("Original")
    plt.axis("off")

    plt.subplot(1, 3, 2)
    plt.imshow(overlay)
    plt.title(title)
    plt.axis("off")

    plt.subplot(1, 3, 3)
    plt.imshow(labels, cmap="nipy_spectral")
    plt.title("Object-label mask")
    plt.axis("off")

    plt.tight_layout()
    plt.show()

# Trial 1 — Automatic SAM segmentation of the entire image

In Trial 1, SAM receives the **entire sea-ice image**.

Students do not provide a point or bounding-box prompt. `SamAutomaticMaskGenerator` samples the image automatically and proposes masks.


## 8. Run SAM on the full image

In [ ]:
mask_generator = make_mask_generator(sam)

start_time = time.time()

with torch.inference_mode():
    masks_full = mask_generator.generate(image)

time_full = time.time() - start_time

print("Trial 1 complete.")
print("Number of masks detected:", len(masks_full))
print(f"Runtime: {time_full:.1f} s")

## 9. Visualize Trial 1

In [ ]:
overlay_full = create_mask_overlay(
    image,
    masks_full,
    alpha=0.62,
    random_seed=42
)

labels_full = masks_to_label_image(
    masks_full,
    image.shape
)

show_three_panel(
    image,
    overlay_full,
    labels_full,
    title=f"Trial 1 — Full-image SAM ({len(masks_full)} masks)"
)

### What does one SAM mask contain?

Each automatically generated region includes information such as:

- `segmentation` — binary mask
- `area` — mask area in pixels
- `bbox` — bounding box
- `predicted_iou` — SAM's estimate of mask quality
- `stability_score` — mask stability


In [ ]:
if len(masks_full) > 0:
    example = masks_full[0]

    print("Keys:", example.keys())
    print()
    print("Area:", example["area"], "pixels")
    print("Bounding box [x, y, width, height]:", example["bbox"])
    print("Predicted IoU:", example["predicted_iou"])
    print("Stability score:", example["stability_score"])

# Trial 2 — Divide the image into an \(N \times N\) grid

Now we test whether segmentation improves when SAM sees **smaller pieces of the same image**.

For example, if

\[
N = 3,
\]

the image is divided into

\[
3 \times 3 = 9
\]

tiles.

SAM runs independently on each tile. Every mask is then mapped back into its original full-image coordinates.

## Why might tiling help?

SAM internally resizes its input before image encoding. In the full image, a small sea-ice floe may become very small after resizing.

With tiling:

- each tile is processed separately;
- a small floe occupies a larger fraction of the model input;
- SAM may detect small floes more successfully.

## Important limitation

A floe that crosses a tile boundary may be artificially split into two masks.

Therefore, tiling may improve small-object detection while also introducing tile-boundary artifacts.


## 10. Choose \(N\)

Change `N` to experiment:

- `N = 2` → 4 tiles
- `N = 3` → 9 tiles
- `N = 4` → 16 tiles

Start with **N = 3**.


In [ ]:
N = 3

assert isinstance(N, int) and N >= 1

print(f"Dividing the image into {N} × {N} = {N*N} tiles.")

## 11. Divide each image dimension into N equal parts

We use `numpy.linspace` to define the boundaries.

If the image width or height is not exactly divisible by `N`, the remaining pixels are distributed among tiles so that **every pixel is included exactly once**.


In [ ]:
x_edges = np.linspace(0, W, N + 1, dtype=int)
y_edges = np.linspace(0, H, N + 1, dtype=int)

print("x boundaries:", x_edges)
print("y boundaries:", y_edges)

plt.figure(figsize=(10, 10))
plt.imshow(image)

for x in x_edges[1:-1]:
    plt.axvline(x=x, linewidth=2)

for y in y_edges[1:-1]:
    plt.axhline(y=y, linewidth=2)

for row in range(N):
    for col in range(N):
        x0, x1 = x_edges[col], x_edges[col + 1]
        y0, y1 = y_edges[row], y_edges[row + 1]

        plt.text(
            (x0 + x1) / 2,
            (y0 + y1) / 2,
            f"({row},{col})",
            ha="center",
            va="center",
            fontsize=12,
            bbox=dict(facecolor="white", alpha=0.7)
        )

plt.title(f"Trial 2 — Image divided into {N} × {N} tiles")
plt.xlim(0, W)
plt.ylim(H, 0)
plt.axis("off")
plt.show()

## 12. Preview the image tiles

In [ ]:
fig, axes = plt.subplots(N, N, figsize=(14, 14))

axes = np.atleast_2d(axes)

for row in range(N):
    for col in range(N):
        x0, x1 = x_edges[col], x_edges[col + 1]
        y0, y1 = y_edges[row], y_edges[row + 1]

        tile = image[y0:y1, x0:x1]

        axes[row, col].imshow(tile)
        axes[row, col].set_title(
            f"Tile ({row},{col})\n"
            f"{tile.shape[1]} × {tile.shape[0]} px"
        )
        axes[row, col].axis("off")

plt.tight_layout()
plt.show()

## 13. Run automatic SAM segmentation independently on every tile

The function below:

1. extracts each tile;
2. runs `SamAutomaticMaskGenerator` on that tile;
3. converts each tile-local mask back to full-image coordinates;
4. preserves a globally located binary mask;
5. records the tile in which each mask was found.

There is still **no manually supplied prompt**.


In [ ]:
def segment_image_by_tiles(image_rgb, model, N):
    """
    Divide an RGB image into N x N non-overlapping tiles,
    run SAM automatic mask generation on each tile,
    and map all local masks back to full-image coordinates.
    """
    height, width = image_rgb.shape[:2]

    x_edges_local = np.linspace(0, width, N + 1, dtype=int)
    y_edges_local = np.linspace(0, height, N + 1, dtype=int)

    generator = make_mask_generator(model)

    all_global_masks = []
    tile_counts = np.zeros((N, N), dtype=int)

    total_tiles = N * N
    tile_number = 0

    for row in range(N):
        for col in range(N):
            tile_number += 1

            x0, x1 = x_edges_local[col], x_edges_local[col + 1]
            y0, y1 = y_edges_local[row], y_edges_local[row + 1]

            tile = image_rgb[y0:y1, x0:x1]

            print(
                f"Tile {tile_number:02d}/{total_tiles}: "
                f"row={row}, col={col}, "
                f"x=[{x0}:{x1}], y=[{y0}:{y1}], "
                f"size={tile.shape[1]}×{tile.shape[0]}"
            )

            with torch.inference_mode():
                local_masks = generator.generate(tile)

            tile_counts[row, col] = len(local_masks)

            for local_mask_info in local_masks:
                local_seg = local_mask_info["segmentation"]

                global_seg = np.zeros(
                    (height, width),
                    dtype=bool
                )

                global_seg[y0:y1, x0:x1] = local_seg

                global_mask_info = dict(local_mask_info)
                global_mask_info["segmentation"] = global_seg

                # Convert tile-local bbox to full-image coordinates.
                bx, by, bw, bh = local_mask_info["bbox"]
                global_mask_info["bbox"] = [
                    bx + x0,
                    by + y0,
                    bw,
                    bh
                ]

                global_mask_info["tile_row"] = row
                global_mask_info["tile_col"] = col

                all_global_masks.append(global_mask_info)

            if device == "cuda":
                torch.cuda.empty_cache()

    return all_global_masks, tile_counts


start_time = time.time()

masks_tiled, tile_counts = segment_image_by_tiles(
    image,
    sam,
    N=N
)

time_tiled = time.time() - start_time

print()
print("Trial 2 complete.")
print("Total number of tiled masks:", len(masks_tiled))
print(f"Runtime: {time_tiled:.1f} s")

## 14. Number of masks detected in each tile

In [ ]:
print("Mask counts by tile:")
print(tile_counts)

plt.figure(figsize=(7, 6))
plt.imshow(tile_counts)

for row in range(N):
    for col in range(N):
        plt.text(
            col,
            row,
            str(tile_counts[row, col]),
            ha="center",
            va="center",
            fontsize=13
        )

plt.title("Number of SAM masks detected in each tile")
plt.xlabel("Tile column")
plt.ylabel("Tile row")
plt.xticks(range(N))
plt.yticks(range(N))
plt.colorbar(label="Number of masks")
plt.show()

## 15. Stitch the tile masks back together and visualize Trial 2

In [ ]:
overlay_tiled = create_mask_overlay(
    image,
    masks_tiled,
    alpha=0.62,
    random_seed=42
)

labels_tiled = masks_to_label_image(
    masks_tiled,
    image.shape
)

show_three_panel(
    image,
    overlay_tiled,
    labels_tiled,
    title=f"Trial 2 — {N}×{N} tiled SAM ({len(masks_tiled)} masks)"
)

# Compare Trial 1 and Trial 2

Now compare:

- **Trial 1:** SAM processes the entire image once.
- **Trial 2:** SAM independently processes \(N \times N\) tiles.

Do **not** assume that more masks automatically means better segmentation. Additional masks may represent real small floes, but they may also represent over-segmentation or tile-boundary artifacts.


## 16. Side-by-side visual comparison

In [ ]:
plt.figure(figsize=(22, 8))

plt.subplot(1, 3, 1)
plt.imshow(image)
plt.title("Original image")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(overlay_full)
plt.title(
    "Trial 1 — Full image\n"
    f"{len(masks_full)} masks"
)
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(overlay_tiled)
plt.title(
    f"Trial 2 — {N}×{N} tiles\n"
    f"{len(masks_tiled)} masks"
)
plt.axis("off")

plt.tight_layout()
plt.show()

## 17. Compare mask counts, mask areas, and runtime

In [ ]:
areas_full = np.array([m["area"] for m in masks_full])
areas_tiled = np.array([m["area"] for m in masks_tiled])

print("TRIAL 1 — FULL IMAGE")
print("--------------------")
print("Number of masks:", len(masks_full))
print(f"Runtime: {time_full:.1f} s")

if len(areas_full) > 0:
    print(f"Median mask area: {np.median(areas_full):.1f} pixels")
    print(f"Smallest mask:    {areas_full.min():.0f} pixels")
    print(f"Largest mask:     {areas_full.max():.0f} pixels")

print()
print(f"TRIAL 2 — {N}×{N} TILES")
print("--------------------")
print("Number of masks:", len(masks_tiled))
print(f"Runtime: {time_tiled:.1f} s")

if len(areas_tiled) > 0:
    print(f"Median mask area: {np.median(areas_tiled):.1f} pixels")
    print(f"Smallest mask:    {areas_tiled.min():.0f} pixels")
    print(f"Largest mask:     {areas_tiled.max():.0f} pixels")

if len(masks_full) > 0:
    change = 100 * (len(masks_tiled) - len(masks_full)) / len(masks_full)
    print()
    print(f"Change in number of masks: {change:+.1f}%")

## 18. Compare physical floe-size distributions (PDF)

SAM reports each mask area in **pixels**. Because the real image width is known, those mask areas can be converted to physical area in square meters.

The image resolution is

\[
\Delta x = \frac{L_x}{W},
\]

where \(L_x=1500\,\mathrm{m}\) is the real image width and \(W\) is the image width in pixels. Assuming square pixels,

\[
A_{\mathrm{pixel}}=(\Delta x)^2.
\]

For a segmented mask containing \(N_p\) pixels, its physical area is

\[
A_f=N_p A_{\mathrm{pixel}}.
\]

We characterize floe size using the **equivalent circular diameter**,

\[
D_f=2\sqrt{\frac{A_f}{\pi}}.
\]

This is the diameter of a circle having the same area as the segmented mask; the actual floe does not need to be circular.

The probability density function \(p(D_f)\) is normalized so that

\[
\int p(D_f)\,dD_f=1.
\]

The same physical pixel scale is used for Trial 1 and Trial 2. Tiling changes how SAM sees the image, but it does **not** change the physical resolution of the original satellite image.

> **Scientific caution:** SAM masks are not automatically guaranteed to be one-to-one physical floes. Overlapping/nested masks and floes cut by tile boundaries can affect the inferred floe-size distribution. The curves below are therefore best interpreted initially as **SAM-detected mask-size distributions** unless the masks are further cleaned and validated.


In [ ]:
# =========================================================
# Physical floe-size distribution (PDF): Trial 1 vs Trial 2
# =========================================================

# SAM's "area" field is measured in pixels in the ORIGINAL image
# coordinates. Trial 1 and Trial 2 therefore use the SAME m/pixel
# conversion even though Trial 2 processes tiles separately.

# Convert mask areas from pixels to square meters.
areas_full_m2 = areas_full * pixel_area_m2
areas_tiled_m2 = areas_tiled * pixel_area_m2

# Equivalent circular diameter [m]:
#
#       A = pi D^2 / 4
#   =>  D = 2 sqrt(A/pi)
#
diameters_full_m = 2.0 * np.sqrt(areas_full_m2 / np.pi)
diameters_tiled_m = 2.0 * np.sqrt(areas_tiled_m2 / np.pi)

# Keep only finite, positive sizes.
D_full = diameters_full_m[
    np.isfinite(diameters_full_m) & (diameters_full_m > 0)
]
D_tiled = diameters_tiled_m[
    np.isfinite(diameters_tiled_m) & (diameters_tiled_m > 0)
]

if len(D_full) == 0 or len(D_tiled) == 0:
    raise RuntimeError(
        "Both trials must contain at least one positive mask before plotting the PDF."
    )

print("PHYSICAL SCALE")
print("--------------")
print(f"Image width:      {IMAGE_WIDTH_M:.2f} m")
print(f"Image height:     {IMAGE_HEIGHT_M:.2f} m")
print(f"Resolution:       {meters_per_pixel:.6f} m/pixel")
print(f"Pixel area:       {pixel_area_m2:.6f} m^2/pixel")

print()
print("TRIAL 1 — FULL IMAGE")
print("--------------------")
print(f"Number of detected masks: {len(D_full)}")
print(f"Minimum D_eq: {D_full.min():.3f} m")
print(f"Median  D_eq: {np.median(D_full):.3f} m")
print(f"Mean    D_eq: {np.mean(D_full):.3f} m")
print(f"Maximum D_eq: {D_full.max():.3f} m")

print()
print(f"TRIAL 2 — {N}×{N} TILES")
print("--------------------")
print(f"Number of detected masks: {len(D_tiled)}")
print(f"Minimum D_eq: {D_tiled.min():.3f} m")
print(f"Median  D_eq: {np.median(D_tiled):.3f} m")
print(f"Mean    D_eq: {np.mean(D_tiled):.3f} m")
print(f"Maximum D_eq: {D_tiled.max():.3f} m")

# ---------------------------------------------------------
# Construct a common PDF for direct comparison
# ---------------------------------------------------------
# Sea-ice floe sizes often span a broad range, so logarithmically
# spaced diameter bins make both small and large detected objects
# visible on the same figure.

all_D = np.concatenate([D_full, D_tiled])
D_min = all_D.min()
D_max = all_D.max()

n_bins = 40

if np.isclose(D_min, D_max):
    # Very unlikely for a real segmentation, but this makes the
    # cell robust if every detected mask has the same size.
    D_min = 0.9 * D_min
    D_max = 1.1 * D_max

bin_edges = np.logspace(
    np.log10(D_min),
    np.log10(D_max),
    n_bins + 1
)

# First count masks in each diameter bin.
counts_full, _ = np.histogram(D_full, bins=bin_edges)
counts_tiled, _ = np.histogram(D_tiled, bins=bin_edges)

bin_widths = np.diff(bin_edges)

# Convert counts to probability density.
# Division by bin width is important because the logarithmic bins
# have unequal widths in meters.
pdf_full = counts_full / (len(D_full) * bin_widths)
pdf_tiled = counts_tiled / (len(D_tiled) * bin_widths)

# Geometric bin centers are appropriate for log-spaced bins.
bin_centers = np.sqrt(bin_edges[:-1] * bin_edges[1:])

# Zero values cannot be shown on a logarithmic y-axis.
valid_full = pdf_full > 0
valid_tiled = pdf_tiled > 0

plt.figure(figsize=(9, 7))

plt.plot(
    bin_centers[valid_full],
    pdf_full[valid_full],
    marker="o",
    linewidth=2,
    markersize=5,
    label="Trial 1: full image"
)

plt.plot(
    bin_centers[valid_tiled],
    pdf_tiled[valid_tiled],
    marker="o",
    linewidth=2,
    markersize=5,
    label=f"Trial 2: {N}×{N} tiles"
)

plt.xscale("log")
plt.yscale("log")
plt.xlabel(r"Equivalent floe diameter, $D_f$ [m]")
plt.ylabel(r"Probability density, $p(D_f)$ [m$^{-1}$]")
plt.title("SAM-Detected Floe-Size Distribution")
plt.grid(True, which="both", alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# Check PDF normalization. The integral should be ~1 for each trial.
integral_full = np.sum(pdf_full * bin_widths)
integral_tiled = np.sum(pdf_tiled * bin_widths)

print()
print("PDF NORMALIZATION CHECK")
print("-----------------------")
print(f"Trial 1 integral = {integral_full:.6f}")
print(f"Trial 2 integral = {integral_tiled:.6f}")


## 19. Zoom into a region to inspect small floes

The full-image view can hide differences. The following cell compares the same zoomed region from the original image, Trial 1, and Trial 2.


In [ ]:
# Fractional coordinates of a region to inspect.
# Change these values to examine another part of the image.
x_fraction_min = 0.25
x_fraction_max = 0.55
y_fraction_min = 0.25
y_fraction_max = 0.55

zx0 = int(W * x_fraction_min)
zx1 = int(W * x_fraction_max)
zy0 = int(H * y_fraction_min)
zy1 = int(H * y_fraction_max)

plt.figure(figsize=(20, 7))

plt.subplot(1, 3, 1)
plt.imshow(image[zy0:zy1, zx0:zx1])
plt.title("Original — zoomed")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(overlay_full[zy0:zy1, zx0:zx1])
plt.title("Trial 1 — full-image SAM")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(overlay_tiled[zy0:zy1, zx0:zx1])
plt.title(f"Trial 2 — {N}×{N} tiled SAM")
plt.axis("off")

plt.tight_layout()
plt.show()

## 20. Inspect possible tile-boundary artifacts

A floe crossing a tile boundary can be segmented independently in the neighboring tiles.

The following figure draws the tile boundaries on top of the stitched Trial 2 result.


In [ ]:
plt.figure(figsize=(11, 11))
plt.imshow(overlay_tiled)

for x in x_edges[1:-1]:
    plt.axvline(x=x, linewidth=2)

for y in y_edges[1:-1]:
    plt.axhline(y=y, linewidth=2)

plt.title(f"Trial 2 segmentation with {N}×{N} tile boundaries")
plt.axis("off")
plt.show()

## 21. Save both segmentation results

In [ ]:
OUTPUT_DIR = "/content/SAM_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

FULL_OVERLAY_PATH = os.path.join(
    OUTPUT_DIR,
    "trial1_full_image_overlay.png"
)
FULL_LABEL_PATH = os.path.join(
    OUTPUT_DIR,
    "trial1_full_image_labels.png"
)

TILED_OVERLAY_PATH = os.path.join(
    OUTPUT_DIR,
    f"trial2_tiled_N{N}_overlay.png"
)
TILED_LABEL_PATH = os.path.join(
    OUTPUT_DIR,
    f"trial2_tiled_N{N}_labels.png"
)

cv2.imwrite(
    FULL_OVERLAY_PATH,
    cv2.cvtColor(overlay_full, cv2.COLOR_RGB2BGR)
)
cv2.imwrite(
    TILED_OVERLAY_PATH,
    cv2.cvtColor(overlay_tiled, cv2.COLOR_RGB2BGR)
)

cv2.imwrite(FULL_LABEL_PATH, labels_full)
cv2.imwrite(TILED_LABEL_PATH, labels_tiled)

print("Saved results:")
print(FULL_OVERLAY_PATH)
print(FULL_LABEL_PATH)
print(TILED_OVERLAY_PATH)
print(TILED_LABEL_PATH)

# Optional experiment: vary N

Try:

```python
N = 2
```

and then:

```python
N = 4
```

Rerun the Trial 2 and comparison cells and consider:

1. Does increasing \(N\) detect more small floes?
2. How does the floe-size PDF change as \(N\) changes?
3. Does a larger \(N\) also increase over-segmentation?
4. Are floes split at tile boundaries, creating artificially small detected objects?
5. How does runtime change?
6. Is the number of masks a sufficient measure of segmentation quality?
7. What ground-truth data would be needed to quantitatively determine which trial is better?


# Discussion — Does SAM "know" sea ice?

SAM was not told:

> "Find sea-ice floes."

It generates visually coherent masks from representations learned during large-scale training.

This experiment illustrates several important ideas for AI in civil and environmental engineering.

### 1. Visual segmentation is not the same as physical understanding

A plausible boundary does not mean SAM understands:

- ice mechanics,
- floe interactions,
- open water,
- melt ponds,
- ridges,
- wave–ice interaction,
- or the physical definition of a sea-ice floe.

### 2. The answer depends on how data are presented

Trial 1 and Trial 2 contain exactly the same physical image, but SAM may give different answers simply because we changed the spatial scale at which the model sees the image.

### 3. More masks do not necessarily mean better masks

Tiling may recover small floes, but it can also introduce:

- fragments of one physical floe,
- water-region masks,
- texture masks,
- and tile-boundary artifacts.

### 4. Engineering validation is still necessary

For scientific or engineering use, segmentation should be evaluated against:

- expert-labeled ground truth,
- physically meaningful quantities,
- uncertainty,
- and domain knowledge.


### 5. Floe-size statistics inherit segmentation errors

A physically meaningful floe-size distribution requires the segmentation to represent individual floes consistently. If SAM splits one floe into several masks, produces nested masks, or cuts a floe at a tile boundary, those segmentation choices propagate directly into the inferred floe-size PDF.
